# Read a TSV file in Python with chDB (drop-in pandas)

Companion to [read-tsv-file-python](https://clickhouse.com/resources/engineering/read-tsv-file-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas`.

## 1. Read a TSV into a DataFrame (types auto-inferred)

In [ ]:
import chdb.datastore as pd

df = pd.read_csv("data/events.tsv", sep="\t")
df

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
revenue = df[df["event_type"] == "purchase"].groupby("country")["revenue"].sum()
revenue

## 3. Headerless TSV: name the columns

In [ ]:
named = pd.read_csv(
    "data/events_noheader.tsv",
    sep="\t",
    names=["event_id", "country", "event_type", "revenue", "product"],
)
named.head(3)

## 4. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 5. Performance: same code, one import swapped, on a 3M-row TSV

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~8.6x faster than `pandas.read_csv`.

In [ ]:
import time

def datastore_agg():
    d = pd.read_csv("data/events_large.tsv", sep="\t")
    r = (d[d["event_type"] == "purchase"]
         .groupby("country")["revenue"].sum()
         .sort_values(ascending=False))
    return r.to_pandas()

def pandas_agg():
    import pandas as real_pd
    p = real_pd.read_csv("data/events_large.tsv", sep="\t")
    return (p[p["event_type"] == "purchase"]
            .groupby("country")["revenue"].sum()
            .sort_values(ascending=False))

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

pd_s = best_of_3(pandas_agg)
ds_s = best_of_3(datastore_agg)
print(f"import pandas as pd:            {pd_s:.3f}s")
print(f"import chdb.datastore as pd:    {ds_s:.3f}s")
print(f"speedup:                        {pd_s / ds_s:.1f}x")